## 빈 components 레코드 제거

`components`가 빈 리스트인 레코드는 재료 정보가 없으므로 제거한다.

In [1]:
from pathlib import Path
import json

GRAPH_INPUT_PATH = next(Path(".").glob("*/recipes_graph_prepared_v2.jsonl"))
GRAPH_OUTPUT_PATH = GRAPH_INPUT_PATH.with_name("recipes_graph_prepared_v2_nonempty.jsonl")

input_records = []
removed_records = 0
with GRAPH_INPUT_PATH.open(encoding="utf-8") as input_file:
    for line_number, line in enumerate(input_file, start=1):
        if not line.strip():
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(f"JSON 파싱 실패: {line_number}번째 줄") from error

        if record.get("components") == []:
            removed_records += 1
            continue
        input_records.append(record)

GRAPH_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with GRAPH_OUTPUT_PATH.open("w", encoding="utf-8") as output_file:
    for record in input_records:
        output_file.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"원본 레코드 수: {len(input_records) + removed_records:,}")
print(f"삭제된 레코드 수: {removed_records:,}")
print(f"남은 레코드 수: {len(input_records):,}")
print(f"저장 경로: {GRAPH_OUTPUT_PATH}")

원본 레코드 수: 9,985
삭제된 레코드 수: 430
남은 레코드 수: 9,555
저장 경로: 전처리\recipes_graph_prepared_v2_nonempty.jsonl


In [2]:
# 결과에 components가 빈 리스트인 레코드가 남아 있지 않은지 검증한다.
with GRAPH_OUTPUT_PATH.open(encoding="utf-8") as output_file:
    saved_records = [json.loads(line) for line in output_file if line.strip()]

assert len(saved_records) == len(input_records)
assert all(record.get("components") != [] for record in saved_records)
print("검증 완료: 빈 components 레코드가 제거되었습니다.")

검증 완료: 빈 components 레코드가 제거되었습니다.


In [2]:
import json
from pathlib import Path

def first_record(path):
    with path.open(encoding="utf-8") as file:
        return json.loads(next(line for line in file if line.strip()))

top_viewed = first_record(Path("홍기표/input/10000recipe_top_viewed.jsonl"))
graph = first_record(Path("전처리/recipes_graph_prepared_v2_nonempty.jsonl"))

print("[10000recipe_top_viewed.jsonl]")
print(list(top_viewed.keys()))
print("\n[recipes_graph_prepared_v2_nonempty.jsonl]")
print("최상위:", list(graph.keys()))
print("recipe:", list(graph["recipe"].keys()))
if graph["components"]:
    print("components.component:", list(graph["components"][0]["component"].keys()))
    print("components.ingredient:", list(graph["components"][0]["ingredient"].keys()))

top_columns = set(top_viewed.keys())
def nested_keys(value):
    keys = set()
    if isinstance(value, dict):
        keys.update(value.keys())
        for child in value.values():
            keys.update(nested_keys(child))
    elif isinstance(value, list):
        for child in value:
            keys.update(nested_keys(child))
    return keys
graph_columns = nested_keys(graph)
print("\n[두 파일의 최상위 컬럼 비교]")
print("남아있는 컬럼:", sorted(top_columns & graph_columns))
print("삭제된 컬럼:", sorted(top_columns - graph_columns))

[10000recipe_top_viewed.jsonl]
['url', 'rank', 'title', 'author', 'views', 'rating_count', 'description', 'ingredients', 'steps', 'categories', 'source', 'collected_at']

[recipes_graph_prepared_v2_nonempty.jsonl]
최상위: ['schema_version', 'recipe', 'dish', 'components']
recipe: ['recipe_uid', 'title', 'source', 'source_url', 'description', 'servings', 'cooking_time', 'difficulty', 'views']
components.component: ['component_uid', 'raw_name', 'role', 'index', 'alternative_mode', 'evidence', 'quality_flags', 'group', 'quantity', 'unit']
components.ingredient: ['name', 'name_normalized']

[두 파일의 최상위 컬럼 비교]
남아있는 컬럼: ['description', 'source', 'title', 'views']
삭제된 컬럼: ['author', 'categories', 'collected_at', 'ingredients', 'rank', 'rating_count', 'steps', 'url']


In [3]:
print(type(collected_at))

NameError: name 'collected_at' is not defined